# 04 — Patna Case Study: Boring Road & Kurji Area

Apply trained models to detect urban change in Patna, Bihar.

**Areas of interest**:
- **Boring Road** (~25.61°N, 85.105°E): Commercial corridor, dense urban development
- **Kurji** (~25.585°N, 85.09°E): Residential expansion, institutional area

In [ ]:
!pip install pystac-client planetary-computer rasterio torchgeo -q

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

if os.path.exists('delta/src'):
    sys.path.insert(0, 'delta')
    print('Using cloned repo')
elif os.path.exists('../src'):
    sys.path.insert(0, '..')
    print('Using parent dir')
else:
    print('Modules not found - check clone step')

import torch
from src.models.siamese_unet import SiameseUNet
from src.models.unet import Autoencoder
from src.data.ndvi import compute_ndvi

print("Imports OK")

## 1. Search Sentinel-2 Imagery via Planetary Computer

In [ ]:
import pystac_client
import planetary_computer

PATNA_AOI = [85.08, 25.57, 85.12, 25.62]  # [west, south, east, north]

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

print("Connected to Microsoft Planetary Computer")

In [ ]:
search_t1 = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=PATNA_AOI,
    datetime="2022-01-01/2022-12-31",
    query={"eo:cloud_cover": {"lt": 10}},
    limit=10,
    sortby=[{"field": "eo:cloud_cover", "direction": "asc"}],
)
t1_items = list(search_t1.items())
print(f"T1 scenes found: {len(t1_items)}")

for item in t1_items[:3]:
    cloud = item.properties.get('eo:cloud_cover', 'N/A')
    date = item.datetime.strftime('%Y-%m-%d')
    print(f"  {date} — Cloud: {cloud:.1f}%")

In [ ]:
search_t2 = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=PATNA_AOI,
    datetime="2024-01-01/2024-12-31",
    query={"eo:cloud_cover": {"lt": 10}},
    limit=10,
    sortby=[{"field": "eo:cloud_cover", "direction": "asc"}],
)
t2_items = list(search_t2.items())
print(f"T2 scenes found: {len(t2_items)}")

for item in t2_items[:3]:
    cloud = item.properties.get('eo:cloud_cover', 'N/A')
    date = item.datetime.strftime('%Y-%m-%d')
    print(f"  {date} — Cloud: {cloud:.1f}%")

## 2. Download and Load Bands

In [ ]:
import rasterio
from rasterio.windows import from_bounds

def load_band(item, band_name, bbox, target_shape=(256, 256)):
    href = item.assets[band_name].href
    with rasterio.open(href) as src:
        window = from_bounds(bbox[0], bbox[1], bbox[2], bbox[3], src.transform)
        data = src.read(1, window=window).astype(np.float32)
    data = np.array(Image.fromarray(data).resize(target_shape, Image.BILINEAR))
    return data

if len(t1_items) > 0 and len(t2_items) > 0:
    best_t1 = t1_items[0]
    best_t2 = t2_items[0]
    print(f"T1: {best_t1.datetime.strftime('%Y-%m-%d')} (cloud: {best_t1.properties['eo:cloud_cover']:.1f}%)")
    print(f"T2: {best_t2.datetime.strftime('%Y-%m-%d')} (cloud: {best_t2.properties['eo:cloud_cover']:.1f}%)")
else:
    print("No scenes found - try wider date range or higher cloud tolerance")
    best_t1 = best_t2 = None

In [ ]:
if best_t1 and best_t2:
    bands_config = ['B04', 'B03', 'B02']  # RGB only
    band_map = {'B04': 'red', 'B03': 'green', 'B02': 'blue'}

    t1_bands = {}
    t2_bands = {}

    for band in bands_config:
        print(f"Loading {band} ({band_map[band]})...")
        t1_bands[band] = load_band(best_t1, band, PATNA_AOI)
        t2_bands[band] = load_band(best_t2, band, PATNA_AOI)

    t1_rgb = np.stack([t1_bands['B04'], t1_bands['B03'], t1_bands['B02']], axis=-1)
    t2_rgb = np.stack([t2_bands['B04'], t2_bands['B03'], t2_bands['B02']], axis=-1)

    t1_rgb_vis = np.clip(t1_rgb / 3000, 0, 1)
    t2_rgb_vis = np.clip(t2_rgb / 3000, 0, 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(t1_rgb_vis)
    axes[0].set_title(f'T1: {best_t1.datetime.strftime("%Y-%m-%d")}')
    axes[0].axis('off')
    axes[1].imshow(t2_rgb_vis)
    axes[1].set_title(f'T2: {best_t2.datetime.strftime("%Y-%m-%d")}')
    axes[1].axis('off')
    plt.suptitle('Patna Sentinel-2 imagery', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Run Supervised Model

In [ ]:
def prepare_model_input_rgb(t1_bands, t2_bands):
    """Prepare RGB-only input tensors matching model's in_channels=3."""
    t1 = np.stack([t1_bands['B04'], t1_bands['B03'], t1_bands['B02']], axis=0).astype(np.float32) / 10000.0
    t2 = np.stack([t2_bands['B04'], t2_bands['B03'], t2_bands['B02']], axis=0).astype(np.float32) / 10000.0
    return torch.from_numpy(t1).unsqueeze(0), torch.from_numpy(t2).unsqueeze(0)

print("Preprocessing function ready")

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
if best_t1 and best_t2:
    t1_tensor, t2_tensor = prepare_model_input_rgb(t1_bands, t2_bands)
    print(f"T1 tensor: {t1_tensor.shape}")
    print(f"T2 tensor: {t2_tensor.shape}")

    checkpoint_sup = 'siamese_unet_best.pt'
    if os.path.exists(checkpoint_sup):
        ckpt = torch.load(checkpoint_sup, map_location=device, weights_only=False)
        cfg = ckpt['config']
        model_sup = SiameseUNet(
            in_channels=cfg['model']['in_channels'],
            out_channels=cfg['model'].get('out_channels', 1),
            features=cfg['model']['features'],
            use_attention=cfg['model'].get('use_attention', False),
        )
        model_sup.load_state_dict(ckpt['model_state_dict'])
        model_sup.to(device).eval()

        with torch.no_grad():
            stacked = torch.cat([t1_tensor, t2_tensor], dim=1).to(device)
            pred_sup = torch.sigmoid(model_sup(stacked)).squeeze().cpu().numpy()

        pred_binary_sup = (pred_sup > 0.5).astype(np.uint8)
        print(f"Supervised prediction: {pred_binary_sup.shape}")
        print(f"Changed pixels: {pred_binary_sup.sum()} / {pred_binary_sup.size} ({pred_binary_sup.mean()*100:.1f}%)")
    else:
        print(f"Checkpoint not found: {checkpoint_sup}")
        print("Upload siamese_unet_best.pt from Colab training (notebook 02)")
        pred_binary_sup = np.zeros((256, 256), dtype=np.uint8)
        pred_sup = np.zeros((256, 256), dtype=np.float32)

## 4. Run Self-Supervised Model

In [ ]:
if best_t1 and best_t2:
    checkpoint_ss = 'autoencoder_best.pt'
    if os.path.exists(checkpoint_ss):
        ckpt_ss = torch.load(checkpoint_ss, map_location=device, weights_only=False)
        cfg_ss = ckpt_ss['config']
        model_ss = Autoencoder(
            in_channels=cfg_ss['model']['in_channels'],
            features=cfg_ss['model']['features'],
        )
        model_ss.load_state_dict(ckpt_ss['model_state_dict'])
        model_ss.to(device).eval()

        with torch.no_grad():
            t1_dev = t1_tensor.to(device)
            t2_dev = t2_tensor.to(device)
            recon_t1 = model_ss(t1_dev)
            recon_t2 = model_ss(t2_dev)

        err_t1 = torch.mean((t1_dev - recon_t1)**2, dim=1).squeeze().cpu().numpy()
        err_t2 = torch.mean((t2_dev - recon_t2)**2, dim=1).squeeze().cpu().numpy()
        change_map_ss = np.abs(err_t1 - err_t2)

        pred_binary_ss = (change_map_ss > 0.15).astype(np.uint8)
        print(f"Self-supervised prediction: {pred_binary_ss.shape}")
        print(f"Changed pixels: {pred_binary_ss.sum()} / {pred_binary_ss.size} ({pred_binary_ss.mean()*100:.1f}%)")
    else:
        print(f"Checkpoint not found: {checkpoint_ss}")
        print("Upload autoencoder_best.pt from Colab training (notebook 03)")
        pred_binary_ss = np.zeros((256, 256), dtype=np.uint8)
        change_map_ss = np.zeros((256, 256), dtype=np.float32)

## 5. Side-by-Side Comparison

In [ ]:
if best_t1 and best_t2:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(t1_rgb_vis)
    axes[0].set_title(f'Before ({best_t1.datetime.strftime("%Y-%m-%d")})')
    axes[0].axis('off')

    axes[1].imshow(t2_rgb_vis)
    axes[1].set_title(f'After ({best_t2.datetime.strftime("%Y-%m-%d")})')
    axes[1].axis('off')

    axes[2].imshow(t2_rgb_vis, alpha=0.6)
    change_rgb = np.zeros((*pred_binary_sup.shape, 3))
    change_rgb[pred_binary_sup > 0] = [1, 0, 0]
    axes[2].imshow(change_rgb, alpha=0.5)
    pct = pred_binary_sup.mean() * 100
    axes[2].set_title(f'Supervised ({pct:.1f}% changed)')
    axes[2].axis('off')

    plt.suptitle('Patna Change Detection', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('patna_supervised_result.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if best_t1 and best_t2 and 'change_map_ss' in dir():
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(t2_rgb_vis)
    axes[0].set_title('After (T2)')
    axes[0].axis('off')

    axes[1].imshow(change_map_ss, cmap='hot')
    axes[1].set_title('Reconstruction Error Map')
    axes[1].axis('off')

    axes[2].imshow(t2_rgb_vis, alpha=0.6)
    change_ss_rgb = np.zeros((*pred_binary_ss.shape, 3))
    change_ss_rgb[pred_binary_ss > 0] = [0, 0, 1]
    axes[2].imshow(change_ss_rgb, alpha=0.5)
    pct_ss = pred_binary_ss.mean() * 100
    axes[2].set_title(f'Self-Supervised ({pct_ss:.1f}% changed)')
    axes[2].axis('off')

    plt.suptitle('Self-Supervised Change Detection — Patna', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('patna_selfsupervised_result.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Statistics

In [ ]:
pixel_size_m = 10  # Sentinel-2 resolution
hectare_per_pixel = (pixel_size_m ** 2) / 10000

methods = {
    'Supervised Siamese U-Net': pred_binary_sup if 'pred_binary_sup' in dir() else np.zeros((256, 256)),
    'Self-Supervised Autoencoder': pred_binary_ss if 'pred_binary_ss' in dir() else np.zeros((256, 256)),
}

print(f"{'Method':<35} {'Changed px':>12} {'Area (ha)':>12} {'% Area':>10}")
print("-" * 70)
for name, mask in methods.items():
    changed = mask.sum()
    area = changed * hectare_per_pixel
    pct = mask.mean() * 100
    print(f"{name:<35} {changed:>12,} {area:>12.2f} {pct:>9.2f}%")

## Summary

- Supervised model: trained on Texas buildings in OSCD, applied to Patna urban fabric
- Self-supervised model: no labels needed, detects any reconstruction anomaly as change
- **Limitation**: OSCD is ~0.5m resolution (Google Earth), Patna Sentinel-2 is 10m. Direct comparison is limited.
- Next: `05_evaluation.ipynb` for detailed metrics and visualizations